# PLN para preprocesamiento de texto 

![Logo Tec](./logo%20tec.png)

**Nombre:**
Amilka Daniela Lopez Aguilar

**Matrícula:**
A01029277

**Materia:**
Inteligencia Artificial Avanzada para la Ciencia de Datos

**Grupo:**
601

**Docente:**
Esteban Castillo Juarez

**Fecha:**
2026-11-8





## Introducción

Este proyecto analiza mensajes de X para identificar si expresan un sentimiento positivo, negativo o neutral. Para que una computadora pueda trabajar con texto, primero hay que convertirlo en números. La forma de hacerlo importa: algunas representaciones conservan mejor las palabras, otras ayudan a reconocer frases o variaciones al escribir (Manning et al., 2008).

Los mensajes de redes sociales suelen ser breves e informales. Además, una frase puede cambiar de sentido por una negación, una broma o el contexto. Por eso no conviene borrar automáticamente signos, emojis, hashtags o palabras como “no”: pueden ayudar a entender el sentimiento.

El análisis parte del preprocesamiento trabajado anteriormente y compara distintas formas de representar los mensajes y entrenar modelos. Después se revisan los resultados para identificar cuáles funcionaron mejor en este conjunto de datos.

También se incluyen diez visualizaciones para conocer mejor los textos: qué palabras aparecen con más frecuencia, cuánto miden los mensajes y qué tan variado es el vocabulario, entre otros aspectos.

### Objetivos

- Comparar distintas formas de limpiar y representar los mensajes.
- Probar varios modelos con las mismas divisiones de entrenamiento y prueba.
- Revisar sus resultados con métricas y visualizaciones, no solo con un número de aciertos.
- Entender qué información del texto ayuda a reconocer el sentimiento.

### Acerca de los datos

El conjunto está compuesto por mensajes de X etiquetados como positivos, negativos o neutrales.

El conjunto original de entrenamiento contiene 4,187 mensajes: 2,249 positivos, 859 negativos y 1,079 neutrales. El conjunto de prueba contiene 867: 358 positivos, 179 negativos y 330 neutrales.

Para algunos experimentos, las etiquetas objective y objective-or-neutral también se agruparon con neutral. Con ese ajuste había 6,165 registros de entrenamiento y 974 de prueba antes de revisar duplicados; después del filtrado, quedaron 6,136 y 966, respectivamente. 

## Palabras clave

Procesamiento de lenguaje natural, análisis de sentimientos, limpieza de texto, clasificación, Naive Bayes, bolsa de palabras, TF-IDF, n-gramas, redes neuronales, regresión logística, Skip-gram, embeddings, BERT y combinación de representaciones.

## Metodología

Primero se revisan y limpian los textos: se normalizan mayúsculas y minúsculas, se tratan enlaces, menciones, hashtags y emojis, y se separan las palabras. Después se prueban distintas formas de convertirlos en características numéricas, como bolsa de palabras, TF-IDF, n-gramas y embeddings.

Por último, se comparan Naive Bayes, regresión logística y redes neuronales. Se mantienen las mismas divisiones de datos para que la comparación sea justa y se utiliza validación cruzada cuando es posible. La comparación no parte de que un modelo sea el mejor; son los resultados los que permiten evaluarlo.

### Modelos incluidos

- Naive Bayes sirve como una opción sencilla y rápida para trabajar con conteos de palabras. Su limitación es que trata las características como si fueran independientes entre sí.
- La regresión logística es un modelo de clasificación que funciona bien con muchas características de texto y permite revisar qué señales influyen en sus predicciones. Con la limitación de que no puede aprender combinaciones complejas de características debido a su naturaleza.
- Las redes neuronales pueden aprender combinaciones más complejas de características, aunque necesitan ajustes cuidadosos para no aprenderse demasiado los ejemplos de entrenamiento, es decir que no generalicen bien a nuevos datos (overfitting).

### Evaluación de los resultados

Además de contar cuántas predicciones fueron correctas, se revisan precisión, recall y F1. Como las clases no tienen la misma cantidad de ejemplos, también se considera el macro-F1, que ayuda a ver si el modelo funciona de manera equilibrada en ambas clases.

### Alcance de esta versión

Esta versión reúne los resultados principales y guarda los modelos seleccionados para poder consultarlos. También resume experimentos que no se muestran uno por uno. Se presenta un Top 20 por tarea: uno para la clasificación binaria y otro para la clasificación de tres clases. Cada lista se ordena con los resultados de su propia tarea, por lo que sus métricas no deben compararse directamente entre sí.


## Preprocesamiento común

Antes de comparar los modelos, los mensajes pasan por una limpieza común. Se corrigen las entidades HTML, se reemplazan enlaces y menciones por marcadores, se conserva el texto de los hashtags, se convierten emojis y emoticonos conocidos en marcadores, se pasa el texto a minúsculas y se separan las palabras. También se expanden contracciones: por ejemplo, “don't” queda como “do not”, para conservar la negación.

Algunas opciones se dejaron como experimentos separados: quitar palabras frecuentes, aplicar stemming o lematización, y unir una negación con la palabra siguiente. Si las aplicáramos desde el inicio, ya no podríamos comparar qué efecto tiene cada una.

El orden de los datos también se mezcla con una semilla fija y cada mensaje se mantiene junto con su etiqueta y grupo. Los conjuntos de entrenamiento y prueba se mezclan por separado y no se combinan. Para este análisis, las etiquetas objective y objective-or-neutral se agrupan con neutral.

Así, todos los modelos parten de una limpieza similar y las diferencias entre ellos se deben principalmente a la representación o al modelo evaluado.


## Resultados anteriores

Antes de esta comparación, ya se habían probado otros métodos. En esos resultados, RoBERTa obtuvo las mejores métricas, por encima de model81, que combina SentiWordNet, TF-IDF y regresión logística. La tabla resume esa comparación anterior; no forma parte del Top 20 de esta nueva ejecución.

| Método | Exactitud macro | Precisión macro | Recall macro | Macro-F1 | ROC-AUC macro |
|---|---:|---:|---:|---:|---:|
| model81: SentiWordNet + TF-IDF + regresión logística | 0.6387 | 0.6175 | 0.6314 | 0.6207 | 0.8134 |
| RoBERTa | 0.7826 | 0.7849 | 0.7702 | 0.7746 | 0.9330 |


## Preprocesamiento por defecto

En la siguiente sección se muestra el proceso de limpieza utilizado como punto de partida para los textos.


In [1]:
import html
import re
from collections import Counter
from sklearn.utils import shuffle

'''
Limpieza textual común para todos los modelos: normalización, expansión de
contracciones y tokenización. Las demás variantes de PLN se configuran dentro
del pipeline para poder compararlas experimentalmente.
'''
SENTIMENT_LABELS = ["negative", "neutral", "positive"]
DATA_SHUFFLE_SEED = 42
DEFAULT_PREPROCESSING_VERSION = "default_sentiment_v2" # Versión por defecto del preprocesamiento de tweets
emoji_markers = {"😊": " emoji_positive ", "😀": " emoji_positive ", "😍": " emoji_positive ", "❤": " emoji_positive ", "❤️": " emoji_positive ", "👍": " emoji_positive ", "😢": " emoji_negative ", "😭": " emoji_negative ", "😡": " emoji_negative ", "👎": " emoji_negative ", "😞": " emoji_negative "}
DEFAULT_URL_PATTERN = re.compile(r"(?:https?://|www\.)\S+", re.IGNORECASE) # Patrón para detectar URLs en el texto
DEFAULT_MENTION_PATTERN = re.compile(r"(?<![A-Za-z0-9_])@[A-Za-z0-9_]+") # Patrón para detectar menciones de usuario en el texto
DEFAULT_EMOTICONS = {":)": "emoji_positive", ":-)": "emoji_positive", ";)": "emoji_positive", ";-)": "emoji_positive", ":D": "emoji_positive", ":-D": "emoji_positive", "=D": "emoji_positive", "<3": "emoji_positive", ":(": "emoji_negative", ":-(": "emoji_negative", ";(": "emoji_negative", ":/": "emoji_neutral", ":\\": "emoji_neutral"}

def preprocess_default_tweet(text):
    '''
    Preprocesa un tweet aplicando la limpieza textual común: decodifica entidades HTML,
    normaliza URLs y menciones con tokens genéricos, conserva el contenido de los hashtags,
    convierte emojis y emoticonos conocidos en marcadores, pasa el texto a minúsculas,
    expande contracciones y tokeniza.
    Args:
        str : El texto del tweet a preprocesar.
    Returns:
        str: El texto preprocesado.
    '''
    text = html.unescape(str(text))
    text = DEFAULT_URL_PATTERN.sub(" urltoken ", text)
    text = DEFAULT_MENTION_PATTERN.sub(" usertoken ", text)
    text = re.sub(r"#([A-Za-z0-9_]+)", r" \1 ", text)
    for emoji in sorted(emoji_markers, key=len, reverse=True):
        text = text.replace(emoji, emoji_markers[emoji])
    for emoticon in sorted(DEFAULT_EMOTICONS, key=len, reverse=True):
        text = text.replace(emoticon, f" {DEFAULT_EMOTICONS[emoticon]} ")
    text = text.casefold()
    text = re.sub(r"\bwon't\b", "will not", text)
    text = re.sub(r"\bcan't\b", "can not", text)
    text = re.sub(r"\b([a-z]+)n't\b", r"\1 not", text)
    return " ".join(re.findall(r"(?u)\b\w+\b", text))

## Revisión de mensajes duplicados

Se comprobó que un mismo mensaje no apareciera tanto en entrenamiento como en prueba. De ocurrir, el modelo podría reconocer un texto que ya vio y parecer más preciso de lo que realmente es. Los duplicados se identificaron comparando los textos y sus identificadores, y se agruparon para mantenerlos juntos. Además, algunos resultados mejoraban de forma inesperada cuando ciertos duplicados quedaban repartidos entre los grupos de validación.


In [2]:
from collections import defaultdict
from pathlib import Path
import pandas as pd

def load_duplicate_demo_records(path, split_name):
    '''
    Carga los registros duplicados de un archivo de datos.
    Args:
        path (Path): La ruta al archivo de datos.
        split_name (str): El nombre del conjunto de datos (por ejemplo, "train" o "test").
    Returns:
        list: Una lista de diccionarios que representan los registros duplicados.
    '''
    rows = []
    with open(path, encoding="utf-8") as dataset_file:
        for line in dataset_file:
            columns = line.rstrip("\r\n").split("\t", maxsplit=3)
            if len(columns) != 4:
                continue
            label = columns[2].strip().casefold()
            if label in {"objective", "objective-or-neutral"}:
                label = "neutral"
            text = columns[3].strip()
            if label in SENTIMENT_LABELS and text:
                rows.append({"tweet_id": columns[0].strip(), "text": text, "label": label, "split": split_name})
    return rows

duplicate_demo_records = load_duplicate_demo_records(Path("originallTraining.txt"), "train") + load_duplicate_demo_records(Path("originalTest.txt"), "test")
duplicate_demo_parents = list(range(len(duplicate_demo_records)))

def duplicate_demo_find(index):
    '''
    Encuentra el grupo representativo de un registro duplicado, para ello, unimos por rango los índices de los registros duplicados.
    Args:
        index (int): El índice del registro duplicado.
    Returns:
        int: El índice del grupo representativo al que pertenece el registro duplicado.
    '''
    while duplicate_demo_parents[index] != index:
        duplicate_demo_parents[index] = duplicate_demo_parents[duplicate_demo_parents[index]]
        index = duplicate_demo_parents[index]
    return index

duplicate_demo_seen = {}
for index, record in enumerate(duplicate_demo_records):
    normalized_text = " ".join(record["text"].casefold().split())
    keys = [f"text:{normalized_text}"]
    if record["tweet_id"]:
        keys.append(f"id:{record['tweet_id']}")
    for key in keys:
        if key in duplicate_demo_seen:
            duplicate_demo_parents[duplicate_demo_find(index)] = duplicate_demo_find(duplicate_demo_seen[key])
        else:
            duplicate_demo_seen[key] = index

duplicate_demo_group_ids = [duplicate_demo_find(index) for index in range(len(duplicate_demo_records))]
duplicate_demo_members = defaultdict(list)
for record, group_id in zip(duplicate_demo_records, duplicate_demo_group_ids):
    duplicate_demo_members[group_id].append(record)
duplicate_demo_repeated_groups = {group_id: members for group_id, members in duplicate_demo_members.items() if len(members) > 1}
duplicate_demo_cross_split_groups = {group_id: members for group_id, members in duplicate_demo_repeated_groups.items() if len({record["split"] for record in members}) > 1}
if len(duplicate_demo_repeated_groups) == 0:
    print("Sin duplicados entre training y testing")
else:
    print(f"Registros leídos: {len(duplicate_demo_records)}")
    print(f"Grupos con duplicados (mismo ID o texto normalizado): {len(duplicate_demo_repeated_groups)}")
    print(f"Registros repetidos agrupados: {sum(len(members) - 1 for members in duplicate_demo_repeated_groups.values())}")
    print(f"Grupos duplicados que aparecen tanto en train como en test: {len(duplicate_demo_cross_split_groups)}")
    duplicate_demo_examples = [{"group_id": group_id, "copies": len(members), "splits": ", ".join(sorted({record["split"] for record in members})), "tweet_id": members[0]["tweet_id"], "text": members[0]["text"][:100]} for group_id, members in list(duplicate_demo_repeated_groups.items())[:5]]
    display(pd.DataFrame(duplicate_demo_examples))

Registros leídos: 7139
Grupos con duplicados (mismo ID o texto normalizado): 67
Registros repetidos agrupados: 67
Grupos duplicados que aparecen tanto en train como en test: 6


,group_id,copies,splits,tweet_id,text
0,19,2,"test, train",264011728084410369,#Broncos Peyton Manning named AFC Offensive Pl...
1,33,2,train,256416307182985217,Capello says time for Russia to win big match:...
2,46,2,train,256787668367704066,Grammys concert pays moving tribute to Whitney...
3,363,2,"test, train",264223821987463169,Blog Post: Vice President Joe Biden to Present...
4,390,2,train,264037153934282752,We're excited for TRUNK OR TREAT tomorrow (Nov...


## Preparación de los conjuntos de datos

Se guardaron dos versiones de los datos. Una conserva las tres clases —negativa, neutral y positiva— y agrupa objective y objective-or-neutral con neutral. La otra es binaria: solo conserva mensajes positivos y negativos, así que deja fuera los neutrales y objetivos.

Esta separación se hizo porque en pruebas anteriores los mensajes neutrales se confundían con los positivos. De esta manera, se puede revisar por separado cómo se comportan los modelos en cada tarea.

![Matriz de confusión de la parte 1](./CM_P1.png)

Los archivos guardados corresponden a entrenamiento y prueba para ambas tareas. Estas versiones se utilizan para comparar las representaciones y los modelos.


In [3]:
from pathlib import Path

original_split_files = {
    "train": Path("originallTraining.txt"),
    "test": Path("originalTest.txt"),
}
dataset_version_files = {
    "3classes": {
        "train": Path("originalTraining_3classes.txt"),
        "test": Path("originalTest_3classes.txt"),
    },
    "binary": {
        "train": Path("originalTraining_binary.txt"),
        "test": Path("originalTest_binary.txt"),
    },
}
three_class_labels = {"negative", "neutral", "positive"}
binary_labels = {"negative", "positive"}

for split_name, source_path in original_split_files.items():
    version_rows = {version: [] for version in dataset_version_files}
    with source_path.open(encoding="utf-8") as source_file:
        for line in source_file:
            columns = line.rstrip("\r\n").split("\t", maxsplit=3)
            if len(columns) != 4 or not columns[3].strip():
                continue
            label = columns[2].strip().casefold()
            if label in {"objective", "objective-or-neutral"}:
                label = "neutral"
            if label not in three_class_labels:
                continue
            columns[2] = label
            version_rows["3classes"].append(columns)
            if label in binary_labels:
                version_rows["binary"].append(columns)

    for version, rows in version_rows.items():
        output_path = dataset_version_files[version][split_name]
        with output_path.open("w", encoding="utf-8", newline="") as output_file:
            for columns in rows:
                output_file.write("\t".join(columns) + "\n")
        print(f"{output_path}: {len(rows)} registros")

originalTraining_3classes.txt: 6165 registros
originalTraining_binary.txt: 3108 registros
originalTest_3classes.txt: 974 registros
originalTest_binary.txt: 537 registros


## Comparación de formas de representar los mensajes

### Diseño de la comparación

La comparación busca ser lo más pareja posible: se mantienen los mismos datos y, cuando se puede, se cambia una sola cosa a la vez.


### Representaciones evaluadas

| Representación | En qué consiste |
|---|---|
| Ensambles | Representación múltiple concatenada y normalizada/estadísticamente transformada |
| Bolsa de palabras | Cuenta cuántas veces aparece cada palabra. Es fácil de entender, aunque no conserva el orden. |
| Presencia o ausencia de palabras | Marca si una palabra aparece o no, sin importar cuántas veces se repite. |
| TF-IDF | Da más importancia a las palabras que ayudan a distinguir un mensaje y menos a las que aparecen por todas partes. |
| N-gramas | Toma grupos seguidos de una, dos o tres palabras. Así puede reconocer expresiones y algunas negaciones. |
| Skip-gram | Aprende vectores de palabras a partir de las palabras que suelen aparecer cerca. Luego, los vectores se combinan para representar cada mensaje. |
| Embeddings propios | Aprende a representar las palabras durante el entrenamiento con los datos disponibles. Puede adaptarse al tema, aunque un conjunto limitado puede dificultar el aprendizaje de palabras poco frecuentes. |
| Embeddings preentrenados | Usa vectores aprendidos previamente con textos de otro conjunto, como GloVe. Así se aprovecha información que el modelo ya había aprendido. |
| BERT | Tiene en cuenta las palabras que rodean a cada término para interpretar su sentido. También se evalúa una versión de BERT ajustada a la tarea. |
| Combinación de representaciones | Une varias formas de representar el mismo mensaje para ver si juntas aportan más información que por separado. En particular, ensemble combina palabras sueltas y pares de palabras con fragmentos de 3 a 5 caracteres, incluidos los espacios entre palabras. |

### Combinación de representaciones

Se probaron las 26 combinaciones posibles de dos o más métodos entre bolsa de palabras, presencia o ausencia, TF-IDF, n-gramas y la representación que combina palabras y caracteres. Se compararon límites de 5,000 y 10,000 características por componente. Así mismo, se prueban distintos hiperparámetros, cada uno con su rango adecuado. Antes de unir los bloques, se llevaron a una escala común para evitar que uno pesara más solo por tener valores más grandes. La escala y la normalización se discuten más adelante. Al final, se muestran los top 20 modelos por cada forma de clasificación de los mensajes (binaria/multiclase).


### Avance de los experimentos

A continuación, algunas combinaciones y representaciones que se generan por defecto y que producen buen desempeño del modelo en F1 y macro exactitud.

- [x] Se generaron las representaciones de palabras, TF-IDF, n-gramas y combinaciones para entrenamiento y prueba.
- [x] Se probó Word2Vec y dos formas de resumir sus palabras en un vector por mensaje.
- [x] Se utilizaron vectores GloVe y BERT previamente entrenados.
- [x] Se compararon el promedio y la concatenación de vectores, con y sin normalización.
- [x] Se compararon configuraciones con validación cruzada, sin usar el conjunto de prueba para escogerlas.
- [x] Se guardaron los vectores de Word2Vec, GloVe y BERT para la tarea binaria y multiclase.
- [x] Se revisaron los mejores resultados de los embeddings con macro-F1. BERT promediado y normalizado obtuvo 0.7635 con C igual a 2; GloVe promediado sin normalizar, 0.7146 con C igual a 0.5; y Word2Vec promediado sin normalizar, 0.6291 con 100 dimensiones y una ventana de 5 palabras.
- [x] Se probaron redes neuronales con distintas representaciones y ajustes, como el tamaño de las capas, la tasa de aprendizaje, el tamaño del lote y el número de épocas.
- [x] Se compararon combinaciones de representaciones con regresión logística y algunas también con una red neuronal.
- [x] Se compararon embeddings de BERT con el modelo base fijo y también una versión ajustada directamente para la tarea binaria.

El siguiente código general las representaciones del texto en vector a partir de una función de python importada para efectos de limpieza en el notebook:


In [4]:
from generate_text_vectors import run_complete_experiments

experiment_results, top20_models = run_complete_experiments(preprocess_default_tweet, top_n=20)
display(top20_models)

Red neuronal en mps
3classes | bow | max_features=5000 | course_64_32_16 | macro-F1=0.5559
3classes | bow | max_features=5000 | wide_128_64_32 | macro-F1=0.5669
3classes | bow | max_features=10000 | course_64_32_16 | macro-F1=0.5559
3classes | bow | max_features=10000 | wide_128_64_32 | macro-F1=0.5669
3classes | one_hot | max_features=5000 | course_64_32_16 | macro-F1=0.5518
3classes | one_hot | max_features=5000 | wide_128_64_32 | macro-F1=0.5623
3classes | one_hot | max_features=10000 | course_64_32_16 | macro-F1=0.5518
3classes | one_hot | max_features=10000 | wide_128_64_32 | macro-F1=0.5623
3classes | tfidf | max_features=5000 | course_64_32_16 | macro-F1=0.5401
3classes | tfidf | max_features=5000 | wide_128_64_32 | macro-F1=0.5577
3classes | tfidf | max_features=10000 | course_64_32_16 | macro-F1=0.5401
3classes | tfidf | max_features=10000 | wide_128_64_32 | macro-F1=0.5577
3classes | ngrams | max_features=5000 | course_64_32_16 | macro-F1=0.5601
3classes | ngrams | max_featur

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


binary | word2vec_mean_l2 | max_features=None | course_64_32_16 | macro-F1=0.5233


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


binary | word2vec_mean_l2 | max_features=None | wide_128_64_32 | macro-F1=0.6074
binary | glove_twitter_25_mean_l2 | max_features=None | course_64_32_16 | macro-F1=0.6970
binary | glove_twitter_25_mean_l2 | max_features=None | wide_128_64_32 | macro-F1=0.7088
binary | bert_base_uncased_mean_l2 | max_features=None | course_64_32_16 | macro-F1=0.7527
binary | bert_base_uncased_mean_l2 | max_features=None | wide_128_64_32 | macro-F1=0.7551


/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6590.70it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.b

,rank,dataset,representation,text_representation_configuration,model_family,algorithm,max_features,hyperparameters,cv_accuracy_mean,cv_accuracy_std,cv_macro_f1_mean,cv_macro_f1_std,test_accuracy,test_macro_f1,model_file
0,1,binary,bert_base_finetuned,"BERT base uncased fine-tuned, max_length=48",bert_finetune,BertForSequenceClassification,NaN,"{""epoch"": 2}",0.896263,0.005467,0.873868,0.006530,0.895131,0.885346,vector_outputs/top20_models/20261101_202735/01...
1,2,binary,bert_base_finetuned,"BERT base uncased fine-tuned, max_length=48",bert_finetune,BertForSequenceClassification,NaN,"{""epoch"": 1}",0.829253,0.032804,0.755293,0.062149,0.898876,0.889441,vector_outputs/top20_models/20261101_202735/02...
2,3,binary,tfidf_word_char,FeatureUnion de TF-IDF word (1-2) + char_wb (3...,classical,LogisticRegression,NaN,"{""C"": 5.0, ""class_weight"": ""none""}",0.810246,0.003555,0.731488,0.005656,0.790262,0.736054,vector_outputs/top20_models/20261101_202735/03...
3,4,binary,tfidf_word_char,FeatureUnion de TF-IDF word (1-2) + char_wb (3...,classical,LogisticRegression,NaN,"{""C"": 5.0, ""class_weight"": ""balanced""}",0.806379,0.003069,0.756459,0.004493,0.784644,0.755271,vector_outputs/top20_models/20261101_202735/04...
4,5,binary,combo_ngrams+ensemble,"Ensamble concatenado de ngrams, ensemble; máxi...",ensemble,MLP,10000.0,"{""batch_size"": 200.0, ""dropout"": 0.2, ""epochs""...",0.803480,0.005884,0.745110,0.008128,0.795880,0.757260,vector_outputs/top20_models/20261101_202735/05...
5,6,binary,tfidf_char,"TF-IDF de caracteres char_wb (3-5), sublinear_...",classical,LogisticRegression,NaN,"{""C"": 5.0, ""class_weight"": ""none""}",0.803159,0.005737,0.716689,0.005316,0.775281,0.708223,vector_outputs/top20_models/20261101_202735/06...
6,7,binary,ngrams,"TF-IDF de unigramas a trigramas, min_df=2; máx...",traditional,LogisticRegression,10000.0,"{""C"": 2.0, ""max_features"": 10000}",0.802193,0.007711,0.755905,0.007448,0.780899,0.758991,vector_outputs/top20_models/20261101_202735/07...
7,8,binary,tfidf_word_char,FeatureUnion de TF-IDF word (1-2) + char_wb (3...,classical,LogisticRegression,NaN,"{""C"": 2.0, ""class_weight"": ""balanced""}",0.801869,0.003866,0.754644,0.006369,0.786517,0.759151,vector_outputs/top20_models/20261101_202735/08...
8,9,binary,tfidf_word,"TF-IDF de palabras (unigramas y bigramas), sub...",classical,LogisticRegression,NaN,"{""C"": 5.0, ""class_weight"": ""balanced""}",0.801548,0.003804,0.752795,0.004254,0.794007,0.768258,vector_outputs/top20_models/20261101_202735/09...
9,10,binary,bow,"CountVectorizer de unigramas, min_df=2; máximo...",traditional,MultinomialNB,5000.0,"{""alpha"": 1.0, ""max_features"": 5000}",0.801547,0.005525,0.724453,0.004783,0.747191,0.690222,vector_outputs/top20_models/20261101_202735/10...


## Top 20 de la clasificación binaria: metodología y definiciones

La tabla reúne las 20 configuraciones con mejores resultados para distinguir mensajes positivos y negativos. El orden se basa en el promedio de validación y macro-F1 se utiliza para desempatar. 

Acerca del ensamble: Es una forma de representar el texto con múltiples formas.

En la tabla, uncased quiere decir que BERT no distingue entre mayúsculas y minúsculas. Fine-tuned quiere decir que el modelo, previamente entrenado con otros textos, se volvió a entrenar con estos mensajes y sus etiquetas para reconocer sentimientos.

En las configuraciones de TF-IDF:

char_wb indica que se toman fragmentos de caracteres dentro de cada palabra, sin cruzar los espacios entre palabras. 

Sublinear_tf hace que repetir una palabra aumente su peso de forma gradual, en vez de aumentarlo en la misma proporción cada vez. 

Min_df=2 significa que una palabra o fragmento debe aparecer en al menos dos mensajes para incluirse en la representación. 

FeatureUnion es el método usado para juntar en una sola representación los resultados de varios transformadores; en la fila 20, por ejemplo, combina el TF-IDF de palabras con el TF-IDF de fragmentos de caracteres.

### Qué incluye cada combinación de representaciones

En las tablas, cada combinación reúne los componentes que aparecen escritos en su nombre. Cada componente se prepara por separado, se ajusta a una escala común y luego se une a los demás. Estos son los componentes:

| Componente | Información que aporta |
|---|---|
| Bolsa de palabras | Frecuencia de cada palabra individual. |
| Presencia o ausencia | Indica si aparece cada palabra, sin importar cuántas veces se repite. |
| TF-IDF | Importancia de cada palabra individual, dando menos peso a las que aparecen en muchos mensajes. |
| N-gramas | TF-IDF de palabras individuales, pares de palabras y grupos de tres palabras consecutivas. |
| Ejemplo de ensamble: Representación de palabras y caracteres | Une TF-IDF de palabras individuales y pares de palabras con TF-IDF de fragmentos de 3 a 5 caracteres del texto. Estos fragmentos pueden incluir espacios entre palabras. |

Por ejemplo, una combinación de TF-IDF y representación de palabras y caracteres contiene tres bloques: TF-IDF de palabras individuales, TF-IDF de pares de palabras y TF-IDF de fragmentos de caracteres. Si además aparece n-gramas, se agrega otro bloque con palabras individuales, pares y grupos de tres.

En el Top 20 binario aparecen estas combinaciones (cada ; representa un delimitador entre ensambles probados): n-gramas con palabras y caracteres; TF-IDF, n-gramas y palabras y caracteres; TF-IDF con palabras y caracteres; presencia o ausencia, n-gramas y palabras y caracteres; y bolsa de palabras, n-gramas y palabras y caracteres.

En el Top 20 de tres clases aparecen estas combinaciones: TF-IDF con palabras y caracteres; bolsa de palabras con palabras y caracteres; n-gramas con palabras y caracteres; presencia o ausencia con palabras y caracteres; bolsa de palabras, TF-IDF y n-gramas con palabras y caracteres; presencia o ausencia, TF-IDF y n-gramas con palabras y caracteres; TF-IDF y n-gramas con palabras y caracteres; bolsa de palabras y n-gramas con palabras y caracteres; bolsa de palabras, presencia o ausencia y n-gramas con palabras y caracteres; y bolsa de palabras, presencia o ausencia, TF-IDF y n-gramas con palabras y caracteres. Las filas que repiten una combinación cambian sus hiperparámetros, como el valor de C o el límite de características.

### Cómo leer las medidas y los límites

L2 es una forma de normalizar un vector: ajusta sus valores para que su longitud total sea 1. Al combinar representaciones, esta operación se aplica a cada bloque por separado para que ninguno domine solo por tener valores de mayor escala. En los embeddings, el promedio se calcula sumando los vectores de las palabras de un mensaje y dividiendo entre la cantidad de palabras; así se obtiene un solo vector que resume el mensaje. En la tabla, el promedio de validación es la media de los resultados obtenidos en los 3 folds anteriormente discutidos en cuanto a validación cruzada. Se incluye macro F1 que a su vez es un promedio del F1-score de cada clase

| # | Representación y configuración | Modelo e hiperparámetros | Acc. CV | Macro-F1 CV | Acc. test | Macro-F1 test |
|---:|---|---|---:|---:|---:|---:|
| 1 | BERT base, sin distinguir mayúsculas, ajustado con estos mensajes y etiquetas; longitud máxima 48 | Clasificación BERT; 2 épocas | 89.82 | 87.56 | 89.51 | 88.53 |
| 2 | BERT base, sin distinguir mayúsculas, ajustado con estos mensajes y etiquetas; longitud máxima 48 | Clasificación BERT; 1 época | 82.51 | 74.51 | 89.89 | 88.94 |
| 3 | TF-IDF palabras (1–2) + char_wb (3–5); sublinear_tf; min_df=2 | Regresión logística; C=5; sin balanceo | 81.02 | 73.15 | 79.03 | 73.61 |
| 4 | TF-IDF palabras (1–2) + char_wb (3–5); sublinear_tf; min_df=2 | Regresión logística; C=5; class_weight=balanced | 80.64 | 75.65 | 78.46 | 75.53 |
| 5 | Ensamble n-gramas + representación ensemble; L2 por bloque; max_features=10000 | MLP 64–32–16; dropout 0.2; lr 0.001; batch 200; 14 épocas | 80.35 | 74.51 | 79.59 | 75.73 |
| 6 | TF-IDF caracteres char_wb (3–5); sublinear_tf; min_df=2 | Regresión logística; C=5; sin balanceo | 80.32 | 71.67 | 77.53 | 70.82 |
| 7 | TF-IDF unigramas–trigramas; min_df=2; max_features=10000 | Regresión logística; C=2 | 80.22 | 75.59 | 78.09 | 75.90 |
| 8 | TF-IDF palabras + caracteres (misma configuración que #3) | Regresión logística; C=2; class_weight=balanced | 80.19 | 75.46 | 78.65 | 75.92 |
| 9 | TF-IDF palabras (1–2); sublinear_tf; min_df=2 | Regresión logística; C=5; class_weight=balanced | 80.15 | 75.28 | 79.40 | 76.83 |
| 10 | BoW unigramas; min_df=2; max_features=5000 | Naive Bayes multinomial; alpha=1 | 80.15 | 72.45 | 74.72 | 69.02 |
| 11 | BoW unigramas; min_df=2; max_features=10000 | Naive Bayes multinomial; alpha=1 | 80.15 | 72.45 | 74.72 | 69.02 |
| 12 | Ensamble TF-IDF + n-gramas + ensemble; L2 por bloque; max_features=10000 | Regresión logística; C=2 | 80.15 | 75.41 | 79.03 | 76.53 |
| 13 | TF-IDF palabras + caracteres (misma configuración que #3) | Regresión logística; C=2; sin balanceo | 80.12 | 70.20 | 78.28 | 71.79 |
| 14 | TF-IDF caracteres char_wb (3–5); sublinear_tf; min_df=2 | Regresión logística; C=5; class_weight=balanced | 80.12 | 75.62 | 78.28 | 75.20 |
| 15 | Ensamble n-gramas + ensemble; L2 por bloque; max_features=10000 | Regresión logística; C=2 | 80.12 | 75.44 | 79.03 | 76.66 |
| 16 | Ensamble TF-IDF + n-gramas + ensemble; L2 por bloque; max_features=5000 | Regresión logística; C=2 | 80.12 | 75.51 | 79.78 | 77.49 |
| 17 | Ensamble TF-IDF + ensemble; L2 por bloque; max_features=10000 | Regresión logística; C=2 | 80.03 | 75.49 | 79.96 | 77.61 |
| 18 | Ensamble one-hot + n-gramas + ensemble; L2 por bloque; max_features=10000 | Regresión logística; C=2 | 79.99 | 75.38 | 80.34 | 77.78 |
| 19 | Ensamble BoW + n-gramas + ensemble; L2 por bloque; max_features=10000 | Regresión logística; C=2 | 79.93 | 75.24 | 80.71 | 78.33 |
| 20 | FeatureUnion TF-IDF palabras (1–2) + caracteres (3–5); max_features=10000 | MLP 64–32–16; dropout 0.2; lr 0.001; batch 200; 14 épocas | 79.93 | 73.66 | 79.21 | 75.54 |

## Top 20 de la clasificación de tres clases/multiclase

Esta lista corresponde a los mensajes negativos, neutrales y positivos. Se ordenó por exactitud media de validación cruzada y macro-F1 como desempate. Se muestran las métricas de validación; las de prueba se reservan para la evaluación final. En esta búsqueda, las 20 primeras configuraciones resultaron ser combinaciones de representaciones evaluadas con regresión logística.

| # | Representaciones combinadas | Modelo y configuración | Exactitud CV | Macro-F1 CV |
|---:|---|---|---:|---:|
| 1 | TF-IDF + representación de palabras y caracteres | Regresión logística; C=1; máximo 10 000 características por componente | 66.80 | 62.24 |
| 2 | Bolsa de palabras + representación de palabras y caracteres | Regresión logística; C=2; máximo 10 000 características por componente | 66.56 | 61.75 |
| 3 | N-gramas + representación de palabras y caracteres | Regresión logística; C=1; máximo 10 000 características por componente | 66.54 | 61.89 |
| 4 | Presencia o ausencia + representación de palabras y caracteres | Regresión logística; C=2; máximo 10 000 características por componente | 66.54 | 61.73 |
| 5 | Bolsa de palabras + TF-IDF + n-gramas + representación de palabras y caracteres | Regresión logística; C=1; máximo 10 000 características por componente | 66.53 | 61.71 |
| 6 | Presencia o ausencia + TF-IDF + n-gramas + representación de palabras y caracteres | Regresión logística; C=1; máximo 10 000 características por componente | 66.53 | 61.80 |
| 7 | TF-IDF + n-gramas + representación de palabras y caracteres | Regresión logística; C=0.5; máximo 10 000 características por componente | 66.51 | 61.99 |
| 8 | Bolsa de palabras + n-gramas + representación de palabras y caracteres | Regresión logística; C=1; máximo 10 000 características por componente | 66.49 | 61.73 |
| 9 | TF-IDF + representación de palabras y caracteres | Regresión logística; C=2; máximo 10 000 características por componente | 66.49 | 61.65 |
| 10 | Bolsa de palabras + presencia o ausencia + n-gramas + representación de palabras y caracteres | Regresión logística; C=1; máximo 10 000 características por componente | 66.48 | 61.78 |
| 11 | TF-IDF + n-gramas + representación de palabras y caracteres | Regresión logística; C=2; máximo 10 000 características por componente | 66.43 | 61.52 |
| 12 | Bolsa de palabras + representación de palabras y caracteres | Regresión logística; C=2; máximo 5 000 características por componente | 66.43 | 61.63 |
| 13 | Bolsa de palabras + presencia o ausencia + n-gramas + representación de palabras y caracteres | Regresión logística; C=2; máximo 10 000 características por componente | 66.38 | 61.40 |
| 14 | Bolsa de palabras + presencia o ausencia + TF-IDF + n-gramas + representación de palabras y caracteres | Regresión logística; C=1; máximo 10 000 características por componente | 66.36 | 61.57 |
| 15 | Bolsa de palabras + TF-IDF + n-gramas + representación de palabras y caracteres | Regresión logística; C=0.5; máximo 10 000 características por componente | 66.35 | 61.72 |
| 16 | TF-IDF + n-gramas + representación de palabras y caracteres | Regresión logística; C=1; máximo 10 000 características por componente | 66.33 | 61.53 |
| 17 | TF-IDF + representación de palabras y caracteres | Regresión logística; C=1; máximo 5 000 características por componente | 66.33 | 61.75 |
| 18 | Bolsa de palabras + TF-IDF + n-gramas + representación de palabras y caracteres | Regresión logística; C=2; máximo 10 000 características por componente | 66.28 | 61.27 |
| 19 | N-gramas + representación de palabras y caracteres | Regresión logística; C=2; máximo 10 000 características por componente | 66.26 | 61.43 |
| 20 | Bolsa de palabras + presencia o ausencia + TF-IDF + n-gramas + representación de palabras y caracteres | Regresión logística; C=2; máximo 10 000 características por componente | 66.26 | 61.31 |

### Qué parece haber ayudado en la tarea binaria

- BERT ajustado para esta tarea quedó claramente por encima de las demás configuraciones en validación cruzada y mantuvo un resultado parecido en la prueba.
- Entre los modelos clásicos, combinar palabras y caracteres dio el mejor resultado de validación por exactitud. Las palabras capturan términos y frases; los fragmentos de caracteres pueden ayudar con distintas formas de escribir una misma palabra.
- Dar más peso a la clase menos frecuente redujo un poco la exactitud, pero mejoró el equilibrio entre clases según macro-F1.
- Algunas combinaciones de representaciones quedaron cerca de los mejores modelos clásicos, aunque agregar más características no garantizó una mejora.
- Las redes neuronales dieron resultados razonables, pero en esta comparación no superaron de manera constante a los mejores modelos clásicos ni a BERT ajustado.

## Análisis de Resultados

### Mensajes positivos y negativos

BERT ajustado para reconocer el sentimiento fue el modelo con mejores resultados generales. Con dos épocas obtuvo 89.82% de exactitud media en validación cruzada y 89.51% en el conjunto de prueba, con un macro-F1 de 88.53%. 

Entre los modelos clásicos, la regresión logística con TF-IDF de palabras y caracteres fue una alternativa sólida: logró 81.02% de exactitud en validación cruzada y 79.03% en prueba. Al dar más peso a la clase menos frecuente, la exactitud bajó un poco, pero el macro-F1 subió. Eso puede ser útil cuando se busca que el modelo trate de forma más pareja los mensajes positivos y negativos.

Combinar representaciones también dio resultados competitivos, pero no siempre mejoró el resultado. BERT usado solo para crear vectores promedio quedó por debajo de BERT ajustado a las etiquetas. Word2Vec promedio fue la alternativa más débil. Una explicación posible es que promediar palabras hace que se pierdan el orden y parte del contexto.

### Mensajes negativos, neutrales y positivos

En la tarea de tres clases, la mejor red neuronal con una combinación de representaciones obtuvo 65.84% de exactitud y 63.10% de macro-F1 en prueba, donde la clase más frecuente representa 44.72% de los datos, es decir la clase positiva, seguida de la negativa. La caída frente a la tarea binaria muestra que distinguir tres sentimientos es más difícil. Es posible que los mensajes neutrales se parezcan a los positivos o negativos.

### Qué ayudó y qué no

- Lo que más ayudó fue ajustar BERT directamente con los ejemplos y etiquetas de la tarea.
- Para una opción más sencilla, la regresión logística con palabras y caracteres dio los mejores resultados entre los modelos clásicos.
- Las combinaciones de representaciones fueron útiles en algunos casos, sobre todo para modelos no ajustados. 
- Los vectores promedio de palabras o de BERT no conservaron tanta información como BERT ajustado a la tarea.

### Gráfico 1. Frecuencia de palabras y ley de Zipf

La curva ordena las palabras del conjunto de entrenamiento de tres clases desde la más frecuente hasta la menos frecuente. La línea discontinua muestra una referencia sencilla para apreciar el patrón de Zipf; se usa el texto ya normalizado.

**Hallazgo clave:*** unas pocas palabras aparecen muchas veces y la frecuencia baja a medida que crece el rango, seguida por una cola larga de palabras menos comunes. Esto ayuda a entender los patrones dentro del vocabulario, donde artículos y palabras más generales pueden estar situadas con alta frecuencia.

In [ ]:
from pathlib import Path
from collections import Counter
import html
import pickle
import re
import sys

import numpy as np
import pandas as pd
import plotly.express as px

search_bases = [Path.cwd(), *Path.cwd().parents]
project_dir = next(
    (candidate for base in search_bases
     for candidate in (base, base / "TC3009" / "PLN" / "01 PLN Techniques")
     if (candidate / "vector_outputs" / "top20_models.csv").is_file()),
    None,
)

results_dir = project_dir / "vector_outputs"
top20 = pd.read_csv(results_dir / "top20_models.csv")
for column in ("rank", "cv_accuracy_mean", "cv_accuracy_std", "cv_macro_f1_mean", "test_accuracy", "test_macro_f1"):
    top20[column] = pd.to_numeric(top20[column], errors="raise")
binary_top = top20.loc[top20["dataset"] == "binary"].sort_values("rank").reset_index(drop=True)
three_top = top20.loc[top20["dataset"] == "3classes"].sort_values("rank").reset_index(drop=True)

emoji_markers = {"😊": " emoji_positive ", "😀": " emoji_positive ", "😍": " emoji_positive ", "❤": " emoji_positive ", "❤️": " emoji_positive ", "👍": " emoji_positive ", "😢": " emoji_negative ", "😭": " emoji_negative ", "😡": " emoji_negative ", "👎": " emoji_negative ", "😞": " emoji_negative "}
def preprocess_for_charts(text):
    text = html.unescape(str(text))
    text = re.sub(r"(?:https?://|www\.)\S+", " urltoken ", text, flags=re.IGNORECASE)
    text = re.sub(r"(?<![A-Za-z0-9_])@[A-Za-z0-9_]+", " usertoken ", text)
    text = re.sub(r"#([A-Za-z0-9_]+)", r" \1 ", text)
    for emoji in sorted(emoji_markers, key=len, reverse=True):
        text = text.replace(emoji, emoji_markers[emoji])
    emoticons = {":)": "emoji_positive", ":-)": "emoji_positive", ";)": "emoji_positive", ";-)": "emoji_positive", ":D": "emoji_positive", ":-D": "emoji_positive", "=D": "emoji_positive", "<3": "emoji_positive", ":(": "emoji_negative", ":-(": "emoji_negative", ";(": "emoji_negative", ":/": "emoji_neutral", ":\\": "emoji_neutral"}
    for emoticon in sorted(emoticons, key=len, reverse=True):
        text = text.replace(emoticon, f" {emoticons[emoticon]} ")
    text = text.casefold()
    text = re.sub(r"\bwon't\b", "will not", text)
    text = re.sub(r"\bcan't\b", "can not", text)
    text = re.sub(r"\b([a-z]+)n't\b", r"\1 not", text)
    return " ".join(re.findall(r"(?u)\b\w+\b", text))

train_path = project_dir / "originalTraining_3classes.txt"
train_texts = []
with train_path.open(encoding="utf-8") as source:
    for line in source:
        fields = line.rstrip("\r\n").split("\t", 3)
        if len(fields) == 4 and fields[2].casefold() in {"negative", "neutral", "positive"} and fields[3].strip():
            train_texts.append(preprocess_for_charts(fields[3]))
word_counts = Counter(token for text in train_texts for token in text.split())
zipf_counts = np.asarray([count for _, count in word_counts.most_common(1000)])
zipf_ranks = np.arange(1, len(zipf_counts) + 1)
zipf_reference = zipf_counts[0] / zipf_ranks
zipf_data = pd.DataFrame({
    "Rango": np.tile(zipf_ranks, 2),
    "Frecuencia": np.r_[zipf_counts, zipf_reference],
    "Serie": ["Observada"] * len(zipf_ranks) + ["Referencia 1/rango"] * len(zipf_ranks),
})

fig = px.line(
    zipf_data, x="Rango", y="Frecuencia", color="Serie", log_x=True, log_y=True,
    title="Frecuencia de palabras y patrón aproximado de Zipf",
    labels={"Rango": "Rango de la palabra", "Frecuencia": "Frecuencia"},
    template="plotly_white",
)
fig.update_traces(line={"width": 2})
fig.update_layout(legend_title_text="", hovermode="x unified")
fig.show()

### Gráfico 2. Relación entre exactitud y Macro-F1

Cada punto representa una configuración del Top 20. La exactitud resume cuántos mensajes se clasificaron correctamente; Macro-F1 promedia el resultado de cada clase y ayuda a ver si el modelo también reconoce las clases menos frecuentes. Las tareas se muestran con símbolos distintos y se interpretan por separado.

**Hallazgo principal:** dentro de cada tarea, ambas medidas se mueven en la misma direcciób. Aun así, los puntos no coinciden exactamente: demostrando junto con la matriz de confusión previamente discutida el rol del desbalance de clases en el desempeño de los modelos y por qué se observa mejor desempeño en la clasificación binaria.

In [ ]:
scatter_data = top20.copy() # copia para poder seguir trabajando sobre un df en los próximos gráficos
scatter_data["tarea"] = scatter_data["dataset"].map({
    "binary": "Clasificación binaria",
    "3classes": "Clasificación de tres clases",
})

fig = px.scatter(
    scatter_data,
    x="cv_accuracy_mean",
    y="cv_macro_f1_mean",
    facet_col="tarea",
    hover_data=["rank", "representation"],
    title="Exactitud y Macro-F1 en los Top 20",
    labels={
        "cv_accuracy_mean": "Exactitud media de validación",
        "cv_macro_f1_mean": "Macro-F1 medio de validación",
    },
    template="plotly_white",
)
fig.update_xaxes(tickformat=".0%")
fig.update_yaxes(tickformat=".0%")
fig.show()

### Gráfico 3. Exactitud de validación de los diez primeros modelos

Las barras ordenan los diez primeros modelos de cada clasificación según la exactitud media de validación cruzada. Cada tarea tiene su propio panel.

**Hallazgo principal:** en la tarea binaria, el primer modelo alcanza 89.63% y el segundo 82.93%; las configuraciones clásicas que siguen se agrupan alrededor de 80–81%. En tres clases, los diez primeros resultados están mucho más juntos, entre 66.48% y 66.80%, por lo que las diferencias entre sus exactitudes son pequeñas. Sin embargo, en ambos métodos se aprecia cómo hay un cierto punto donde cambios en los modelos, por más agresivos que sean, no incrementan significativamente los resultados.

In [ ]:
bar_data = pd.concat(
    [binary_top.head(10), three_top.head(10)],
    ignore_index=True,
)
bar_data["tarea"] = bar_data["dataset"].map({
    "binary": "Clasificación binaria",
    "3classes": "Clasificación de tres clases",
})
bar_data["modelo"] = "Modelo " + bar_data["rank"].astype(int).astype(str)

fig = px.bar(
    bar_data,
    x="cv_accuracy_mean",
    y="modelo",
    orientation="h",
    facet_col="tarea",
    category_orders={"modelo": [f"Modelo {rank}" for rank in range(10, 0, -1)]},
    hover_data=["representation"],
    title="Exactitud de validación de los diez primeros modelos",
    labels={
        "cv_accuracy_mean": "Exactitud media de validación",
        "modelo": "Posición",
    },
    template="plotly_white",
)
fig.update_xaxes(tickformat=".0%", matches=None)
fig.update_yaxes(matches=None)
fig.show()

### Gráfico 4. Variación de la exactitud entre particiones o folds

Cada caja resume la desviación estándar de exactitud reportada para los 20 modelos de cada tarea. Una caja más baja y compacta indica que los resultados variaron menos entre las particiones de validación cruzada; los puntos individuales representan modelos.

**Hallazgo principal:** la mediana de variación es de aproximadamente 0.51 puntos porcentuales en binaria y 1.18 en tres clases. En esta búsqueda, los resultados de tres clases cambiaron más entre particiones. Por ende, probablemente notamos las caídas en exactitud cuando promediamos e incluimos el ruido que genera la clase con menos peso y con menos precisión: la neutra.

In [ ]:
variability = top20.assign(
    tarea=top20["dataset"].map({
        "binary": "Clasificación binaria",
        "3classes": "Clasificación de tres clases",
    }),
    desviacion_pp=top20["cv_accuracy_std"] * 100,
)

fig = px.box(
    variability,
    x="tarea",
    y="desviacion_pp",
    points="outliers",
    title="Variación de exactitud entre particiones de validación",
    labels={
        "tarea": "Tarea",
        "desviacion_pp": "Desviación estándar (puntos porcentuales)",
    },
    template="plotly_white",
)
fig.show()

### Gráfico 5. Matriz de confusión del mejor modelo de tres clases

La matriz cuenta las predicciones del modelo que quedó en primer lugar para tres clases sobre el conjunto de prueba. 

**Hallazgo principal:** el modelo reconoce mejor los mensajes neutrales (72.5% de recuperación) que los negativos (53.1%) y positivos (64.8%). La clase negativa es la que más se confunde, sobre todo con neutral.

In [ ]:
import torch
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

model_row = three_top.iloc[0]
model_path = project_dir / model_row["model_file"]
payload = torch.load(model_path, map_location="cpu", weights_only=False)
labels = payload["metadata"]["labels"]
model_bundle = pickle.loads(bytes(payload["pipeline_pickle"].tolist()))
metadata_path = results_dir / "3classes" / "test_metadata.csv"
test_metadata = pd.read_csv(metadata_path, dtype={"tweet_id": str})
raw_test_texts = {}
with (project_dir / "originalTest_3classes.txt").open(encoding="utf-8") as source:
    for line in source:
        fields = line.rstrip("\r\n").split("\t", 3)
        if len(fields) == 4 and fields[3].strip():
            raw_test_texts[fields[0].strip()] = fields[3].strip()
texts = [preprocess_for_charts(raw_test_texts[tweet_id]) for tweet_id in test_metadata["tweet_id"].astype(str)]
sys.path.insert(0, str(project_dir))
from generate_text_vectors import _transform_text_representation
features = _transform_text_representation(model_bundle["vectorizer_bundle"], texts)
predictions = model_bundle["estimator"].predict(features)
label_ids = {label: index for index, label in enumerate(labels)}
y_true = np.asarray([label_ids[label] for label in test_metadata["label"]])
confusion = confusion_matrix(y_true, predictions, labels=np.arange(len(labels)))
observed_accuracy = accuracy_score(y_true, predictions)
observed_macro_f1 = f1_score(y_true, predictions, average="macro")
confusion_df = pd.DataFrame(confusion, index=[label.capitalize() for label in labels], columns=[label.capitalize() for label in labels])
confusion_df.index.name = "Etiqueta real"
confusion_df.columns.name = "Predicción"
fig = px.imshow(
    confusion_df, text_auto=True, color_continuous_scale="Blues", aspect="auto",
    title="Matriz de confusión del primer modelo de tres clases",
    labels={"x": "Predicción", "y": "Etiqueta real", "color": "Mensajes"},
    template="plotly_white",
)
fig.update_layout(coloraxis_colorbar_title="Mensajes")
print(f"Exactitud comprobada: {observed_accuracy:.4f}; Macro-F1 comprobado: {observed_macro_f1:.4f}")
fig.show()

Exactitud comprobada: 0.6605; Macro-F1 comprobado: 0.6351


### Gráfico 6. Diferencia entre prueba y validación

Para cada modelo se calcula la exactitud de prueba menos la exactitud media de validación cruzada. Un valor positivo indica que la prueba quedó por encima del promedio de validación; uno negativo indica lo contrario. 

**Hallazgo principal:** la diferencia mediana es de −1.11 puntos porcentuales en binaria y −0.31 en tres clases. 

In [ ]:
gap_data = top20.assign(
    tarea=top20["dataset"].map({
        "binary": "Clasificación binaria",
        "3classes": "Clasificación de tres clases",
    }),
    diferencia_pp=(top20["test_accuracy"] - top20["cv_accuracy_mean"]) * 100,
)

fig = px.histogram(
    gap_data,
    x="diferencia_pp",
    color="tarea",
    barmode="overlay",
    opacity=0.7,
    title="Diferencia entre prueba y validación cruzada",
    labels={
        "diferencia_pp": "Diferencia de exactitud (puntos porcentuales)",
        "count": "Cantidad de modelos",
    },
    template="plotly_white",
)
fig.add_vline(x=0, line_dash="dash", line_color="black")
fig.show()

### Gráfico 7. Distribución de Macro-F1 en prueba

Cada gráfico muestra cómo se distribuye Macro-F1 de prueba entre los 20 modelos de una tarea. La marca central indica la mediana y cada punto representa una configuración.

**Hallazgo principal:** los resultados binarios abarcan un rango amplio, de 69.02% a 88.94%, porque el grupo incluye configuraciones con comportamientos muy distintos. Los resultados de tres clases se concentran entre 63.24% y 65.47%. Lo que coincide con entregas de este ejercicio anteriores donde el modelo no aumentaba en desempeño a partir de cierto punto.

In [ ]:
violin_data = top20.assign(
    tarea=top20["dataset"].map({
        "binary": "Clasificación binaria",
        "3classes": "Clasificación de tres clases",
    }),
    macro_f1_test_pct=top20["test_macro_f1"] * 100,
)

fig = px.violin(
    violin_data,
    x="tarea",
    y="macro_f1_test_pct",
    box=True,
    title="Distribución de Macro-F1 en prueba",
    labels={
        "tarea": "Tarea",
        "macro_f1_test_pct": "Macro-F1 en prueba (%)",
    },
    template="plotly_white",
)
fig.show()

### Gráfico 8. Perfil de F1 por clase de los mejores modelos

Los diagramas radiales muestran el F1 de cada clase para el primer modelo de cada tarea. Los valores más cercanos al borde representan un resultado más alto para esa clase.

**Hallazgo principal:** en binaria, el F1 positivo (91.9%) supera al negativo (85.2%). En tres clases, el F1 negativo (52.8%) queda por debajo de neutral (70.4%) y positivo (67.3%). Así se identifica con claridad que los mensajes negativos siguen siendo el punto débil del mejor modelo de tres clases.

In [ ]:
binary_class_metrics = pd.read_csv(
    results_dir / "bert_finetune_binary_test_results.csv"
).iloc[0]
three_class_metrics = pd.read_csv(
    results_dir / "ensemble_combo_test_results.csv"
)
three_class_metrics = three_class_metrics.loc[
    three_class_metrics["representation"] == three_top.iloc[0]["representation"]
].iloc[0]

class_scores = pd.DataFrame([
    {"tarea": "Binaria", "clase": "Negativo", "F1": binary_class_metrics["f1_negative"]},
    {"tarea": "Binaria", "clase": "Positivo", "F1": binary_class_metrics["f1_positive"]},
    {"tarea": "Tres clases", "clase": "Negativo", "F1": three_class_metrics["f1_negative"]},
    {"tarea": "Tres clases", "clase": "Neutral", "F1": three_class_metrics["f1_neutral"]},
    {"tarea": "Tres clases", "clase": "Positivo", "F1": three_class_metrics["f1_positive"]},
])

fig = px.scatter(
    class_scores,
    x="F1",
    y="clase",
    color="tarea",
    facet_col="tarea",
    category_orders={"clase": ["Negativo", "Neutral", "Positivo"]},
    title="F1 por clase en los mejores modelos",
    labels={"F1": "F1 en prueba", "clase": "Clase", "tarea": "Tarea"},
    template="plotly_white",
)
fig.update_xaxes(range=[0, 1], tickformat=".0%", matches=None)
fig.update_layout(showlegend=False)
fig.show()

### Gráfico 9. Macro-F1 de validación a lo largo del ranking

Cada punto muestra el Macro-F1 medio de validación cruzada para una posición del Top 20. Las líneas verticales conectan cada valor con la base del gráfico, lo que permite observar tanto el orden como las distancias entre modelos.

**Hallazgo principal:**  en binaria, el primer modelo destaca frente al grupo que le sigue; en tres clases, los Macro-F1 de los primeros lugares se mantienen en una franja estrecha. Esto refuerza que la primera posición no siempre supone una mejora amplia respecto de las configuraciones siguientes.

In [ ]:
rank_data = top20.assign(
    tarea=top20["dataset"].map({
        "binary": "Clasificación binaria",
        "3classes": "Clasificación de tres clases",
    }),
    macro_f1_cv_pct=top20["cv_macro_f1_mean"] * 100,
)

fig = px.line(
    rank_data,
    x="rank",
    y="macro_f1_cv_pct",
    color="tarea",
    facet_col="tarea",
    markers=True,
    title="Macro-F1 de validación según el puesto",
    labels={
        "rank": "Puesto en el ranking",
        "macro_f1_cv_pct": "Macro-F1 de validación (%)",
    },
    template="plotly_white",
)
fig.update_xaxes(dtick=2, matches=None)
fig.update_yaxes(matches=None)
fig.show()

### Gráfico 10. Exactitud de validación y prueba en los diez primeros

El diagrama conecta, para cada modelo, la exactitud media de validación cruzada con la exactitud de prueba. Los dos extremos permiten ver si la prueba quedó por encima o por debajo de la validación.

**Hallazgo principal:** la mayoría de los modelos muestra una exactitud de prueba algo menor que su promedio de validación. Hay excepciones, como el segundo modelo binario, cuyo resultado de prueba sube con claridad. Esa diferencia ilustra por qué la selección se apoya en validación cruzada, pues nos permite evaluar sobre una base con mejor capacidad de generalización.

In [ ]:
comparison = top20.assign(
    tarea=top20["dataset"].map({
        "binary": "Clasificación binaria",
        "3classes": "Clasificación de tres clases",
    }),
)

fig = px.scatter(
    comparison,
    x="cv_accuracy_mean",
    y="test_accuracy",
    color="tarea",
    hover_data=["rank", "representation"],
    title="Exactitud de validación y prueba",
    labels={
        "cv_accuracy_mean": "Exactitud media de validación",
        "test_accuracy": "Exactitud en prueba",
        "tarea": "Tarea",
    },
    template="plotly_white",
)
fig.add_shape(
    type="line",
    x0=0.6,
    y0=0.6,
    x1=1,
    y1=1,
    line={"dash": "dash", "color": "gray"},
)
fig.update_xaxes(range=[0.6, 1], tickformat=".0%")
fig.update_yaxes(range=[0.6, 1], tickformat=".0%")
fig.show()

## Conclusiones

En la clasificación binaria, BERT ajustado para la tarea obtuvo el mejor resultado general: alcanzó 89.63% de exactitud media en validación cruzada y 89.51% en prueba, con un macro-F1 de 88.53% en prueba. 

En la clasificación de tres clases, la mejor configuración fue la regresión logística con TF-IDF y la representación de palabras y caracteres (char_wb), que es uno de los ensambles propuestos para esta tarea. Obtuvo 66.80% de exactitud media y 62.24% de macro-F1 en validación cruzada; en prueba alcanzó 66.05% de exactitud y 63.51% de macro-F1. El resultado fue menos equilibrado para los mensajes negativos: su F1 fue 52.8%, frente a 70.4% para los neutrales y 67.3% para los positivos. Por tanto, reconocer los mensajes negativos es el principal aspecto que requiere mejorar en esta tarea.

En conjunto, las pruebas muestran que combinar representaciones puede dar resultados competitivos al capturar mejor la información del texto y su contexto asociado. En tres clases, las configuraciones mejor posicionadas tuvieron resultados cercanos entre sí, mientras que en binaria BERT mantuvo una ventaja clara.

## Referencias
Las siguientes referencias fueron utilizadas para recabar ideas de gráficos más representativos del modelo que a su vez permitan tener hallazgos más interesantes:

Choque, G. (2026). Medium. Medium. https://medium.com/@gladyschoqueulloa20/los-12-gr%C3%A1ficos-m%C3%A1s-importantes-en-ciencia-de-datos-9fd99f57d1c0

Prabhakaran, S. (2018, November 28). Top 50 matplotlib visualizations – the master plots (with full python code). Machinelearningplus. https://machinelearningplus.com/plots/top-50-matplotlib-visualizations-the-master-plots-python/

Las siguientes referencias fueron utilizadas para teoría y definiciones:

IBM. (2025, 20 de agosto). ¿Qué es el análisis de sentimiento? https://www.ibm.com/mx-es/think/topics/sentiment-analysis

Manning, C. D., Raghavan, P., & Schütze, H. (2008). Introduction to information retrieval. Cambridge University Press. https://nlp.stanford.edu/IR-book/

Alg0.dev. (s. f.). LRU cache: Estructuras de datos. https://www.alg0.dev/es/lru-cache/

NLTK Project. (2025, 1 de octubre). Sample usage for SentiWordNet. NLTK 3.9.2 documentation. https://www.nltk.org/howto/sentiwordnet.html

cardiffnlp/twitter-roberta-base-sentiment-latest · Hugging Face. (2024, January 4). https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest

AI Spectrum. (2024, Septiembre 15). Twitter sentiment analysis for 100+ languages with roBERTa [Video]. YouTube. https://www.youtube.com/watch?v=t_A_35m9OzU

GeeksforGeeks. (2022, Octubre 12). NLP | Synsets for a word in WordNet. GeeksforGeeks. https://www.geeksforgeeks.org/nlp/nlp-synsets-for-a-word-in-wordnet/

Material de clase. (s. f.). PLN4 [PDF], secciones 7.2–8.2, pp. 21–25.